# Labor forecast workflow conversion

Incremental pandas translation of the Alteryx workflow. This notebook currently covers the **Current Labor Forecast** input through the first formula after Transpose. Later branches and database outputs are still to be added as their configurations are captured.

## Current Labor Forecast input

Alteryx Dynamic Input (2) reads the `Data - Forecast & Opps` sheet from the current workbook, treats row 1 as headers, and adds `FileName` as the file name only. The workbook path comes from the existing `.env` key `currentlaborpath`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
current_labor_path = Path(os.environ["currentlaborpath"])
current_labor_raw = pd.read_excel(
    current_labor_path,
    sheet_name="Data - Forecast & Opps",
    header=0,
)
current_labor_raw["FileName"] = current_labor_path.name
print(len(current_labor_raw), "current labor forecast rows loaded")

## GLC cleanup and Transpose (93)

The Transpose tool keeps the eight descriptive fields and `FileName`, excludes `Labor Forecast Total`, and transposes the selected monthly columns plus dynamic or unknown columns. Missing expected fields raise a clear error for review.

In [ ]:
current_labor_raw["GLC"] = current_labor_raw["GLC"].astype("string").str.strip()

key_columns = [
    "Type", "Program", "Program Description", "Project Family",
    "Lane", "Segment", "Firm/FO-SS/New", "GLC", "FileName",
]
missing_keys = [column for column in key_columns if column not in current_labor_raw.columns]
if missing_keys:
    raise KeyError(f"Missing current forecast key columns: {missing_keys}")

value_columns = [
    column for column in current_labor_raw.columns
    if column not in key_columns and column != "Labor Forecast Total"
]
current_labor = current_labor_raw.melt(
    id_vars=key_columns,
    value_vars=value_columns,
    var_name="Name",
    value_name="Value",
)
print(len(current_labor), "transposed rows from", len(value_columns), "data columns")

## Formula (95)

`DoesHaveMonth` uses `REGEX_Match([Name], ".*\\d+.*")` to identify a digit in the transposed column name. `Segment` uses `Trim([Segment])` and has a 255-character output field.

In [ ]:
current_labor["DoesHaveMonth"] = (
    current_labor["Name"].astype("string").str.contains(r"\d+", regex=True, na=False)
)
current_labor["Segment"] = current_labor["Segment"].astype("string").str.strip().str.slice(0, 255)
current_labor[["Name", "Value", "DoesHaveMonth", "Segment"]].head()